In [1]:
import mlflow
import mlflow.sklearn
import os
import pickle as pkl
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score

# ==========================================
# [1] MLflow 초기 설정
# ==========================================
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# ★ 핵심 수정: 실험 이름을 바꿔서 새 경로(Windows 경로)를 잡도록 강제함
experiment_name = "agent10_tone_param_adjust_v2" 
experiment = mlflow.set_experiment(experiment_name)

print(f"✅ 새로운 실험 생성(또는 로드): {experiment.experiment_id}")
print(f"   저장 경로: {experiment.artifact_location}")

if mlflow.active_run():
    mlflow.end_run()

# ==========================================
# [2] 데이터 로드 (자동 경로 탐색)
# ==========================================
candidate_paths = [
    "data_csv", "data_csv/",               
    "../data_csv", "../data_csv/",            
    "../../data_csv", "../../data_csv/",         
    "../../../data_csv", "../../../data_csv/"       
]

target_dir = None

for path in candidate_paths:
    test_path = os.path.join(path, "tone_vectors.pkl")
    if os.path.exists(test_path):
        target_dir = path
        print(f"📍 폴더를 찾았습니다: {target_dir}")
        break

if target_dir is None:
    raise FileNotFoundError("❌ data_csv 폴더를 못 찾았습니다.")

PATH_VECTORS = os.path.join(target_dir, "tone_vectors.pkl")
PATH_META = os.path.join(target_dir, "tone_metadata.csv")

# 파일명 자동 보정
PATH_PROFILE = os.path.join(target_dir, "tone_centroid_profile.csv") 
if not os.path.exists(PATH_PROFILE):
    PATH_PROFILE = os.path.join(target_dir, "tone_profile_template.csv")

print(f"📂 로드할 프로필 파일: {PATH_PROFILE}")

with open(PATH_VECTORS, "rb") as f:
    tone_dict = pkl.load(f)

tone_ids = pd.read_csv(PATH_META)["tone_id"].tolist()
tone_param_df = pd.read_csv(PATH_PROFILE)

rows = []

# ==========================================
# [3] 데이터 전처리 & 숫자 변환(Mapping)
# ==========================================
level_map = {
    'High': 3, 'high': 3, 'HIGH': 3,
    'Mid': 2,  'mid': 2,  'MID': 2, 'Medium': 2,
    'Low': 1,  'low': 1,  'LOW': 1
}

for tone_id in tone_ids:
    if tone_id not in tone_dict:
        continue
        
    vec = tone_dict[tone_id]
    
    subset = tone_param_df[tone_param_df["tone_id"] == tone_id]
    if subset.empty:
        continue
        
    params = subset.iloc[0]

    # 숫자 변환 적용
    p_level = level_map.get(params["proof_level"], 0)
    e_level = level_map.get(params["emotion_level"], 0)
    c_strength = level_map.get(params["cta_strength"], 0)

    row = {
        "tone_id": tone_id,
        **{f"v_{j}": vec[j] for j in range(len(vec))},
        "proof_level": p_level,
        "emotion_level": e_level,
        "cta_strength": c_strength,
        "sentence_len": params["sentence_len"]
    }
    rows.append(row)

df = pd.DataFrame(rows)
print(f"✅ 데이터프레임 생성 완료: {df.shape}")

X = df.filter(regex="^v_").values
y_proof   = df["proof_level"].values
y_emotion = df["emotion_level"].values
y_cta     = df["cta_strength"].values

# ==========================================
# [4] 모델 학습
# ==========================================

# 1. Ridge (Regression)
with mlflow.start_run(run_name="ridge_regression", experiment_id=experiment.experiment_id):
    model = Ridge(alpha=1.0)
    model.fit(X, y_proof)
    
    preds = model.predict(X)
    rmse = np.sqrt(mean_squared_error(y_proof, preds))
    
    mlflow.log_metric("rmse_proof", rmse)
    mlflow.sklearn.log_model(model, "ridge_proof")
    print(f"✅ Ridge RMSE (Proof Level): {rmse:.4f}")

# 2. Logistic (Classification)
with mlflow.start_run(run_name="logistic_emotion", experiment_id=experiment.experiment_id):
    y_emotion_int = y_emotion.astype(int)

    if len(np.unique(y_emotion_int)) < 2:
        print("⚠️ 클래스가 1개뿐입니다.")
    
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X, y_emotion_int)
    
    preds = clf.predict(X)
    acc = accuracy_score(y_emotion_int, preds)
    
    mlflow.log_metric("acc_emotion", acc)
    mlflow.sklearn.log_model(clf, "logistic_emotion")
    print(f"✅ Logistic Accuracy (Emotion Level): {acc:.4f}")

print("\n🚀 [완전 성공] 경로 문제 해결됨 & 학습 완료!")

2025/12/24 14:34:56 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/24 14:34:56 INFO mlflow.store.db.utils: Updating database tables
2025/12/24 14:34:56 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/24 14:34:56 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/24 14:34:56 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/24 14:34:56 INFO alembic.runtime.migration: Will assume non-transactional DDL.


✅ 새로운 실험 생성(또는 로드): 2
   저장 경로: file:///c:/STUDY-DATA/third_week/12_24/mlruns/2
📍 폴더를 찾았습니다: ../data_csv
📂 로드할 프로필 파일: ../data_csv\tone_centroid_profile.csv
✅ 데이터프레임 생성 완료: (4, 773)


2025/12/24 14:34:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/12/24 14:35:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Ridge RMSE (Proof Level): 0.8292
✅ Logistic Accuracy (Emotion Level): 0.5000

🚀 [완전 성공] 경로 문제 해결됨 & 학습 완료!
